# 🧠 AI Agents

This notebook demonstrates how to build a **LangChain agent** with multiple tools.

---

## 🔍 What is an Agent?

An **agent** is like an intelligent assistant that:
1. Thinks about the user's request (**Thought**)
2. Decides what tool(s) to use (**Action**)
3. Executes the tool(s) and observes results (**Observation**)
4. Returns the final response (**Final Answer**)

This is based on the **ReAct framework** (Reason + Act), which gives LLMs structured reasoning steps.

This means that ReAct agents can:
- **Reason**: The model thinks about the problem.
- **Act**: It chooses and uses a tool.
- **Observe** : It sees the tool’s result and continues.

---

## 🔧 What are Tools?

- Tools are **functions** (APIs, calculators, web search, etc.) that the agent can call.
- Each tool has:
  - A **name**
  - A **description** (so the agent knows when to use it)
  - A **function** that does the actual work


In [1]:
from dotenv import load_dotenv
import warnings
from langchain_groq import ChatGroq

In [2]:
# load environment variables from .env file
load_dotenv()

True

### Different Models which can be used

[Here](https://console.groq.com/docs/deprecations) you will find all the models available other than which are listed below.

| Model                   | Size / Type        | Context Window | Strengths                                                                 | Best Use Cases                                      |
|--------------------------|-------------------|----------------|---------------------------------------------------------------------------|----------------------------------------------------|
| **gemma2-9b-it**         | 9B, instruction-tuned | 8K             | Google’s Gemma v2 instruction-tuned model. Good at following prompts, lightweight compared to 70B models. | Small-to-mid agents, lightweight reasoning, fine for simple tool use with prompting. |
| **llama-3.3-70b-versatile** | 70B, general versatile | 8K             | Meta’s flagship Llama 3.3 model hosted on Groq. High reasoning quality, supports structured outputs and function calling. | Complex agents, production-grade apps, when you need reliability and accuracy. |
| **llama-3.1-8b-instant** (not working in this example)   | 8B, fast & compact | 128K           | Optimized for **speed** with large 128K context window. Supports JSON mode and function calling, very low latency. | Real-time apps, chatbots, smaller agents where fast responses matter. |


In [3]:
# Create an LLM instance and select the model we want to use
llm_model = ChatGroq(
    model="llama-3.3-70b-versatile",        #you can try other models if you wish
    temperature=0,
    max_tokens=512,
)

### Defining Tools 🔨
We want to equip our agent with 2 simple custome-made tools:
1. **MathTool**: Solves math expressions using Python's `eval()`.
2. **LengthTool**: Counts the number of characters in a string.

In [4]:
from langchain.tools import Tool
from langchain.agents import initialize_agent, AgentType


##### Custome-made Tool 1

In [5]:
# Tool 1

# Define the function the tool will run
def math_calculator(input_string: str) -> str:
    """A simple calculator tool that performs arithmetic operations."""
    result = eval(input_string)                     # eval() interprets a valid Python math expression given as a string and executes it, returning the result
    return f"The result is: {result}"

In [6]:
# Wrap the function as a LangChain Tool
math_tool = Tool(
    name="MathTool",                    # arbitrary tool name; the agent will call this name in its actions
    func=math_calculator,               # the function the agent is allowed to execute
    description="Use this tool to solve basic math problems. Input should be a valid math expression like '8 * 12'."  # guidance for the agent
)


##### Custom Tool 2

In [7]:
# Tool 2

# Define the function the tool will run
def string_length(input_string: str) -> str:
    """A tool to count the number of characters in a string."""
    length = len(input_string)
    return f"The length of the string is: {length} characters."

In [8]:
# Wrap the function as a LangChain Tool
length_tool = Tool(
    name="LengthTool",
    func=string_length,
    description="Use this tool to count the number of characters in a string."
)

> __Very important__:
> 
>  Add __clear descriptions__ for each tools 🛠️!!!
> 
> The agent **doesn't know how the tools work internally**.  
> 
> It relies entirely on the tool's **name** and **description** to decide when to use them.

### Initialize the Agent
We create a type of agent called `zero-shot-react-description`, which means:
- It uses the **ReAct** reasoning framework:
  - **Thought**: Decide what to do
  - **Action**: Pick a tool
  - **Observation**: See the tool's output
  - Repeat until it has the **Final Answer**
- **Zero-shot** means the agent doesn't need any examples.  
  It figures out how to use tools **just from their descriptions**.

##### 🧠 LangChain Agent Types — Overview

| AgentType | Description | Best For / Use Case |
|-----------|-------------|---------------------|
| `AgentType.ZERO_SHOT_REACT_DESCRIPTION` | Classic ReAct-style agent. Chooses tools by reasoning step-by-step from scratch using only descriptions. | Simple tool use, when you want the agent to figure out how to solve tasks on its own. |
| `AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION` | Same as above, but optimized for chat models (e.g., gpt-3.5-turbo). Uses `ChatPromptTemplate`. | Chat-based apps with tool use. |
| `AgentType.REACT_DOCSTORE` | Uses ReAct logic, but built for interacting with a single knowledge base (like Wikipedia). | Agents that need to read docs or search a single data source. |
| `AgentType.SELF_ASK_WITH_SEARCH` | Specialized agent for answering factual questions. Agent asks itself sub-questions, then answers them using a search tool. | Fact-based Q&A (like "What year did X happen?"). Often paired with web search. |
| `AgentType.CONVERSATIONAL_REACT_DESCRIPTION` | Like `ZERO_SHOT_REACT_DESCRIPTION`, but keeps track of conversation history (with memory). | Conversational agents using tools. Good for chatbots. |
| `AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION` | Chat version of the above. Uses `ChatPromptTemplate` with memory and tool reasoning. | Best for multi-turn chats + tools. |
| `AgentType.OPENAI_FUNCTIONS` | Uses OpenAI's function calling (like GPT-4 tools API). LLM chooses tool via function schema. | If you're using OpenAI and want tools selected via native function-calling interface. |
| `AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION` | Newer ReAct-like chat agent that formats output as JSON (structured actions). Works well with function schemas. | When you want tool calls to be structured and parsable (e.g., APIs, apps). |
| `AgentType.OPENAI_MULTI_FUNCTIONS` | Same as `OPENAI_FUNCTIONS`, but allows calling **multiple tools at once**. | Advanced OpenAI workflows needing multi-function execution. |


In [9]:
# list attributes inside the AgentType class.
dir(AgentType)[:9]

['CHAT_CONVERSATIONAL_REACT_DESCRIPTION',
 'CHAT_ZERO_SHOT_REACT_DESCRIPTION',
 'CONVERSATIONAL_REACT_DESCRIPTION',
 'OPENAI_FUNCTIONS',
 'OPENAI_MULTI_FUNCTIONS',
 'REACT_DOCSTORE',
 'SELF_ASK_WITH_SEARCH',
 'STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION',
 'ZERO_SHOT_REACT_DESCRIPTION']

In LangChain, an AgentType is a preset agent recipe:
- It defines how the prompt is structured (system text, instructions, where tools are listed).
- It defines how the LLM should express its reasoning and tool calls (ReAct style, function-calling style, chat style, etc.).
- It defines how the loop works (thought → action → observation → final answer).

They do not change the model itself. They just change the protocol the LLM follows to reason and call tools.

You always provide:
1. the LLM (e.g. ChatGroq(...))
2. the tools
3. the AgentType

For now, you only really need this level:
- ZERO_SHOT_REACT_DESCRIPTION → ReAct-style, single-turn, uses tool descriptions to decide what to do. Good default when experimenting with tools.
- CONVERSATIONAL_REACT_DESCRIPTION → same ReAct idea, but with conversation history (memory) for multi-turn chats.
- The CHAT_* versions → same logic as above, but tuned for “chat-style” prompts (chat templates) instead of plain text.
- The OPENAI_* ones → same idea, but use OpenAI’s function-calling style instead of ReAct text (“call this function with these args”).
- The docstore / search ones → specialized versions where the recipe assumes “I mostly talk to a document store or search tool.”

In [10]:
langchain_agent = initialize_agent(
    tools=[math_tool, length_tool], # List of tools to use
    llm=llm_model, # The language model to use
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION, # Type of agent to create
    verbose=True, # Whether to print out the agent's reasoning
    max_iterations=3, # Maximum number of iterations for the agent
    return_intermediate_steps=False, # Whether to return intermediate steps
)

# Initialize an agent that can use our tools
langchain_agent = initialize_agent(
    tools=[math_tool, length_tool],                     # the tools the agent is allowed to call
    llm=llm_model,                                      # the LLM that drives the agent's reasoning
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,   # This one uses ReAct-style reasoning without examples
    verbose=True,                                       # print the agent's thoughts, actions, and observations
    max_iterations=3,                                   # stop after 3 tool-use cycles to avoid infinite loops
    return_intermediate_steps=False,                    # only return the final answer, not the reasoning trace
)


/Users/alastair/Github/ds-ai-agent/.venv/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.3.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(


### Run the Agent 🤖

In [11]:
# Example 1: Math question
print("---- Math Question ----")
response = langchain_agent.invoke("What is 15 * 4?")        ## Run the chain with the given input and capture the output it returns
print("✅ Final Answer:", response)

---- Math Question ----


> Entering new AgentExecutor chain...
Thought: To find the answer, I need to multiply 15 by 4, which is a basic math operation. I can use the MathTool to perform this calculation.
Action: MathTool
Action Input: 15 * 4
Observation: The result is: 60
Thought:I now know the final answer
Final Answer: 60

> Finished chain.
✅ Final Answer: {'input': 'What is 15 * 4?', 'output': '60'}


The output shows the agent's chain of reasoning:
+ thought: reasoning steps
+ action: which tool it chose
+ observation: the result of the tool call

Notice that the **agent** turned a natural language prompt ("What is 15 * 4?") into a python maths string ("15 * 4") because it knows that the tool uses the eval() method.

An Agent does two things the LLM alone does not do:

1. It interprets the user query. It looks at your natural-language question and decides: 
    - “Does this require a tool?”
    - “Which tool?”
    - “What input should I give to that tool?”
2. It rewrites the user query into a tool-friendly format

In [12]:
# Example 2: String length question
print("\n---- String Length Question ----")
response = langchain_agent.invoke("How many characters are in the string '' ?")
print("✅ Final Answer:", response['output'])



---- String Length Question ----


> Entering new AgentExecutor chain...
Thought: To find the number of characters in the given string, I should use a tool that can count characters. Since the string is empty, it's likely that the count will be 0, but I should still use the appropriate tool to confirm this.

Action: LengthTool
Action Input: 
Observation: The length of the string is: 0 characters.
Thought:Thought: I now know the final answer
Final Answer: 0

> Finished chain.
✅ Final Answer: 0


The ReAct protocol forces the model to "think" according to:
- Thought:
- Action:
- Action Input:
- Observation:
 
I find it interesting that the thought is objectively incorrect. "To find the number of characters in the given string, I need to count them" is, quite clearly, not how this agent is going to come to an answer. Obviously this is a massive flaw in the system and probably explains why so many of them are so shit (e.g. ChatGPT is actually incapable of understanding this point cause it's fucking stupid).

## Summary
- Agents are intelligent assistants that can think, act, and observe.
- They use tools to perform tasks.
- LangChain provides a framework for building agents with structured reasoning.
- You can create custom tools and define how the agent should use them.